In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from spark_config import SparkConfig
from logger import LoggerConfig
from io_config import IOConfig
from platform_app import PlatformApp

In [10]:
logger = LoggerConfig().setup_logger(log_dir=IOConfig.LOG_DIR)
spark = SparkConfig.create_spark(app_name="SQL in Spark", logger=logger, use_databricks=True)
app = PlatformApp(spark=spark, logger=logger, catalog_name="databricks_toturial")

2026-03-05 14:36:38 | INFO     | Logging configured: level=DEBUG, format=colored
2026-03-05 14:37:04 | INFO     | Connected to Databricks via Spark Connect.
2026-03-05 14:37:04 | INFO     | Initializing Data Platform...
2026-03-05 14:37:04 | INFO     | Spark session initialized


### Create Dataframes

In [12]:
from pyspark.sql import functions as F, types as T

rows_customers = [
    (1,  "Asha",  "IN", True),
    (2,  "Bob",   "US", False),
    (3,  "Chen",  "CN", True),
    (4,  "Diana", "US", None),
    (None, "Ghost","UK", False),     # NULL key to demo null join behavior
]

rows_orders = [
    (101, 1,   120.0, "IN"),
    (102, 1,    80.0, "IN"),
    (103, 2,    50.0, "US"),
    (104, 5,    30.0, "DE"),         # no matching customer_id
    (105, 3,   200.0, "CN"),
    (106, None, 15.0, "UK"),         # NULL key won’t match
    (107, 3,    40.0, "CN"),
    (108, 2,    75.0, "US"),
]

schema_customers = T.StructType([
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("name",        T.StringType(),  True),
    T.StructField("country",     T.StringType(),  True),
    T.StructField("vip",         T.BooleanType(), True),
])

schema_orders = T.StructType([
    T.StructField("order_id",    T.IntegerType(), True),
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("amount",      T.DoubleType(),  True),
    T.StructField("country",     T.StringType(),  True),  # same column name to show collisions
])

df_customers = spark.createDataFrame(rows_customers, schema_customers)
df_orders    = spark.createDataFrame(rows_orders,    schema_orders)

display(df_customers)
display(df_orders)

,customer_id,name,country,vip
0,1.0,Asha,IN,True
1,2.0,Bob,US,False
2,3.0,Chen,CN,True
3,4.0,Diana,US,None
4,NaN,Ghost,UK,False


,order_id,customer_id,amount,country
0,101,1.0,120.0,IN
1,102,1.0,80.0,IN
2,103,2.0,50.0,US
3,104,5.0,30.0,DE
4,105,3.0,200.0,CN
5,106,NaN,15.0,UK
6,107,3.0,40.0,CN
7,108,2.0,75.0,US


### Inner Join

In [13]:
df_inner = df_orders.join(df_customers, on="customer_id", how="inner")
df_inner.show(truncate=False)

+-----------+--------+------+-------+----+-------+-----+
|customer_id|order_id|amount|country|name|country|vip  |
+-----------+--------+------+-------+----+-------+-----+
|1          |101     |120.0 |IN     |Asha|IN     |true |
|1          |102     |80.0  |IN     |Asha|IN     |true |
|2          |103     |50.0  |US     |Bob |US     |false|
|3          |105     |200.0 |CN     |Chen|CN     |true |
|3          |107     |40.0  |CN     |Chen|CN     |true |
|2          |108     |75.0  |US     |Bob |US     |false|
+-----------+--------+------+-------+----+-------+-----+



In [14]:
o = df_orders.alias("o")
o.show(truncate=False)

+--------+-----------+------+-------+
|order_id|customer_id|amount|country|
+--------+-----------+------+-------+
|101     |1          |120.0 |IN     |
|102     |1          |80.0  |IN     |
|103     |2          |50.0  |US     |
|104     |5          |30.0  |DE     |
|105     |3          |200.0 |CN     |
|106     |NULL       |15.0  |UK     |
|107     |3          |40.0  |CN     |
|108     |2          |75.0  |US     |
+--------+-----------+------+-------+



In [15]:
o, c = df_orders.alias("o"), df_customers.alias("c")
df_inner = o.join(c, on="customer_id", how="inner")
df_inner.show(truncate=False)

+-----------+--------+------+-------+----+-------+-----+
|customer_id|order_id|amount|country|name|country|vip  |
+-----------+--------+------+-------+----+-------+-----+
|1          |101     |120.0 |IN     |Asha|IN     |true |
|1          |102     |80.0  |IN     |Asha|IN     |true |
|2          |103     |50.0  |US     |Bob |US     |false|
|3          |105     |200.0 |CN     |Chen|CN     |true |
|3          |107     |40.0  |CN     |Chen|CN     |true |
|2          |108     |75.0  |US     |Bob |US     |false|
+-----------+--------+------+-------+----+-------+-----+



### Disambiguate Columns

In [16]:
df_inner_clean = (
    o.join(c, on="customer_id", how="inner")
     .select(
        "order_id", 
        "customer_id", 
        "amount",
        F.col("o.country").alias("ship_country"),
        "name", 
        F.col("c.country").alias("cust_country"), 
        "vip"
     )
)
df_inner_clean.show(truncate=False)

+--------+-----------+------+------------+----+------------+-----+
|order_id|customer_id|amount|ship_country|name|cust_country|vip  |
+--------+-----------+------+------------+----+------------+-----+
|101     |1          |120.0 |IN          |Asha|IN          |true |
|102     |1          |80.0  |IN          |Asha|IN          |true |
|103     |2          |50.0  |US          |Bob |US          |false|
|105     |3          |200.0 |CN          |Chen|CN          |true |
|107     |3          |40.0  |CN          |Chen|CN          |true |
|108     |2          |75.0  |US          |Bob |US          |false|
+--------+-----------+------+------------+----+------------+-----+



### Other Joins: Left, Full etc.

In [17]:
df_left = o.join(c, on="customer_id", how="left")
df_left.show(truncate=False)

+-----------+--------+------+-------+----+-------+-----+
|customer_id|order_id|amount|country|name|country|vip  |
+-----------+--------+------+-------+----+-------+-----+
|1          |101     |120.0 |IN     |Asha|IN     |true |
|1          |102     |80.0  |IN     |Asha|IN     |true |
|2          |103     |50.0  |US     |Bob |US     |false|
|5          |104     |30.0  |DE     |NULL|NULL   |NULL |
|3          |105     |200.0 |CN     |Chen|CN     |true |
|NULL       |106     |15.0  |UK     |NULL|NULL   |NULL |
|3          |107     |40.0  |CN     |Chen|CN     |true |
|2          |108     |75.0  |US     |Bob |US     |false|
+-----------+--------+------+-------+----+-------+-----+



In [18]:
df_full = o.join(c, on="customer_id", how="full")
df_full.show(truncate=False)

+-----------+--------+------+-------+-----+-------+-----+
|customer_id|order_id|amount|country|name |country|vip  |
+-----------+--------+------+-------+-----+-------+-----+
|1          |101     |120.0 |IN     |Asha |IN     |true |
|1          |102     |80.0  |IN     |Asha |IN     |true |
|2          |103     |50.0  |US     |Bob  |US     |false|
|5          |104     |30.0  |DE     |NULL |NULL   |NULL |
|3          |105     |200.0 |CN     |Chen |CN     |true |
|NULL       |106     |15.0  |UK     |NULL |NULL   |NULL |
|3          |107     |40.0  |CN     |Chen |CN     |true |
|2          |108     |75.0  |US     |Bob  |US     |false|
|NULL       |NULL    |NULL  |NULL   |Ghost|UK     |false|
|4          |NULL    |NULL  |NULL   |Diana|US     |NULL |
+-----------+--------+------+-------+-----+-------+-----+



### Left Semi and Left Anti

In [19]:
df_left_semi = o.join(c, on="customer_id", how="left_semi")
df_left_semi.show(truncate=False)

+-----------+--------+------+-------+
|customer_id|order_id|amount|country|
+-----------+--------+------+-------+
|1          |101     |120.0 |IN     |
|1          |102     |80.0  |IN     |
|2          |103     |50.0  |US     |
|3          |105     |200.0 |CN     |
|3          |107     |40.0  |CN     |
|2          |108     |75.0  |US     |
+-----------+--------+------+-------+



In [20]:
df_left_anti = o.join(c, on="customer_id", how="left_anti")
df_left_anti.show(truncate=False)

+-----------+--------+------+-------+
|customer_id|order_id|amount|country|
+-----------+--------+------+-------+
|5          |104     |30.0  |DE     |
|NULL       |106     |15.0  |UK     |
+-----------+--------+------+-------+



### Multi Key Join

In [21]:
df_multi = o.join(c, on=["customer_id", "country"], how="inner")
df_multi.show(truncate=False)

+-----------+-------+--------+------+----+-----+
|customer_id|country|order_id|amount|name|vip  |
+-----------+-------+--------+------+----+-----+
|1          |IN     |101     |120.0 |Asha|true |
|1          |IN     |102     |80.0  |Asha|true |
|2          |US     |103     |50.0  |Bob |false|
|3          |CN     |105     |200.0 |Chen|true |
|3          |CN     |107     |40.0  |Chen|true |
|2          |US     |108     |75.0  |Bob |false|
+-----------+-------+--------+------+----+-----+

